# Microsoft Agent Framework — Azure OpenAI (Responses API)

ఈ కోడ్ నమూనాలో, మీరు **Microsoft Agent Framework (MAF)** ఉపయోగించి **Azure OpenAI** మద్దతుతో సరళమైన ఏజెంట్‌ను **Responses API** ద్వారా సృష్టించబడుతుంది.

> **మైగ్రేషన్ గమనిక:** ఈ నమూనా మునుపు Semantic Kernel మరియు GitHub Models ఉపయోగించింది. ఇది Microsoft Agent Framework కి మార్చబడింది, GitHub Models (పదవీ విరమణ 2026 జూలై) స్థానంలో Azure OpenAI వచ్చింది, ఇది Responses API ను మద్దతు ఇస్తుంది. MAFలోని `OpenAIChatClient` Azure OpenAI యొక్క స్థిరమైన `/openai/v1/` ఎండ్‌పాయింట్‌ను లక్ష్యంగా పెట్టి డిఫాల్ట్‌గా Responses API ఉ్ఫయోగిస్తుంది.

ఈ నమూనా ఉద్దేశ్యం భవిష్యత్తులో వివిధ ఏజెంట్ నమూనాలను అమలు చేసే అదనపు కోడ్ నమూనాలలో అనుసరించే దశలను చూపించడమే.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## అవసరమైన పైథాన్ ప్యాకేజీలను దిగుమతి చేసుకోండి


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## ఒక టూల్ ని నిర్వచించడం

Microsoft Agent Framework లో, **టూల్** అనేది `@tool` తో అలంకరించిన సాదా Python ఫంక్షన్, దీన్ని ఏజెంట్ పిలవవచ్చు. కింద మనం ఒక టూల్ ని నిర్వచిస్తాము ఇది ఒక యాదృచ్ఛిక సెలవుల గమ్యస్థానాన్ని ఇవ్వడం మరియు గతాన్ని మళ్లీ పునరావృతం కాకుండా నిరోధిస్తుంది.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## ఏజెంట్ సృష్టించడం

ఇక్కడ, మనం `TravelAgent` అనే ఏజెంట్ ను సృష్టించుకుంటున్నాము.

ఈ ఉదాహరణలో, మనం చాలా బేసిక్ సూచనలను ఉపయోగిస్తున్నాము. ఏజెంట్ ప్రవర్తన ఎలా మారుతుందో అర్థం చేసుకోవడానికి మీరు ఈ సూచనలను ఆవిర్భావించడానికి స్వేచ్ఛగా మోడిఫై చేసుకోండి.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## ఏజెంట్ను నడపడం  

ఇప్పుడు మనం ఏజెంట్ను నడుపగలము. ఏజెంట్ సంభాషణను మళ్లీ మళ్లీ గుర్తుంచుకుంటూ జరుగుతుండడానికి మేము ఒక `AgentSession` ని సృష్టిస్తాము, ఆపై రెండు `user_inputs` ను పంపుతాము. మొదటి ప్రయాణం గురించి అడుగుతుంది; రెండవది యూజర్ ఆ సిఫార్సు ఇష్టం కాలేదని, మరొకటి అడుగుతుంది — ఏజెంట్ సెషన్ చరిత్రను మరియు `get_random_destination` సాధనాన్ని ఉపయోగించి ప్రతిస్పందిస్తుంది.  

మీరు ఈ సందేశాలను మార్చి ఏజెంట్ ఎలా వ్యతిరేకంగా స్పందిస్తుందో చూడవచ్చు. ప్రతిస్పందనలు **టోకెన్-ద్వారా టోకెన్** ప్రసారమవుతాయి.  


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
